In [2]:
//#r "BoSSSpad/BoSSSpad.dll"
#r "../../src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

MPI Rank 0: Value for OMP_PLACES: 
MPI Rank 0: Value for OMP_PROC_BIND: 
Number of CPUs in system: 24
not defined: OMP_NUM_THREADS
System reports 24 CPUs, 1 MPI ranks on current node.
Failed to determine user wish for number of threads; trying to use all! System reports 24 CPUs, will use all but 2 for BoSSS (22 total, 22 per MPI rank, MPI ranks on current node is 1).
Finally, setting number of OpenMP and Parallel Task Library threads to 22
CCP_AFFINITY not set
R0: reserved CPUs: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], C# reports mask FFFFFF
R0: reserved CPUs for this MPI rank: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], on entire SMP: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], disjount CPU affinities? False
MpiJobOwnsEntireComputer = True, RnkJobOwnsEntireComputer = True
R0: using CPUs [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23] for OpenMP.
R0: TPL thread pinning: False, OMP thread pinning: False
R0

Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 263
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 105
   at Submission#2.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [3]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [4]:
BoSSSshell.WorkflowMgm.Init("RisingBubble3D", myBatch);

Project name is set to 'RisingBubble3D'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\RisingBubble3D'.


In [5]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [6]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;

## Sessions

In [7]:
bool restartRun = false;

In [8]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_BDF3_sC_testcase1_withOutputs_restart1*	11/20/2024 3:32:33 PM	984fc798...
#1: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_singleCore_testcase1*	9/10/2024 12:38:07 PM	cf93bc52...
#2: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_fixedInterfaceCheck_testcase1*	9/5/2024 3:13:05 PM	00b7639e...
#3: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1*	9/5/2024 8:59:21 AM	d25fc934...
#4: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_withPreEnforcerActive_restart1*	8/28/2024 3:41:52 PM	a929ee94...
#5: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1*	8/9/2024 11:54:11 AM	cb26cb13...
#6: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_restart1*	8/5/2024 9:45:57 AM	9d1f82be...
#7: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_setttingsCheck2	8/1/2024 9:27:13 AM	eff0e8c8...
#8: RisingBubble3D	RB3D_16x16x32AMR1_k2_testcase1_setttingsCheck	4/19/2024 4:25:45 PM	19291211...


In [9]:
//sessions.Pick(0).Delete(true); //.Export().WithSupersampling(2).Do()

In [10]:
var restartSession = sessions.Pick(2);
restartSession

RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_fixedInterfaceCheck_testcase1*	9/5/2024 3:13:05 PM	00b7639e...

In [11]:
//restartSession.Timesteps.Skip(0).Export().WithSupersampling(3).Do()

In [12]:
string restartName = "RB3D_fullDomain_8x8x16AMR0_k3_BDF3_testcase1_withOutputs_restart1";

## Grid Creation

In [13]:
bool fullDomain = true;

In [14]:
int[] Resolutions = new int[] { 8 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"RisingBubble3D_{Res}x{Res}x{2*Res}";
    GridName = fullDomain ? GridName + "_fullDomain" : "_quarterDomain";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = fullDomain ? GenericBlas.Linspace(0, 1.0, Res + 1) : GenericBlas.Linspace(0.5, 1.0, Res + 1);
        double[] yNodes = xNodes;
        double[] zNodes = GenericBlas.Linspace(0, 2.0, Res*2 + 1);
        
        var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;

            if (fullDomain) {
                if(X.x.Abs() <= 1e-8 || X.y.Abs() <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            } else {
                if((X.x - 0.5).Abs() <= 1e-8 || (X.y - 0.5).Abs() <= 1.0e-8)
                ret = IncompressibleBcType.SlipSymmetry.ToString();
            }
            if((X.x - 1.0).Abs() <= 1e-8 || (X.y - 1.0).Abs() <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if((X.z).Abs() <= 1e-8)         // lower wall
                ret = IncompressibleBcType.Wall.ToString();
            if((X.z - 2.0).Abs() <= 1e-8)   // upper wall
                ret = IncompressibleBcType.Wall.ToString();

            return ret;
        });    
        
        // grd.AddPredefinedPartitioning("FourProcSplit_fullDomain", delegate (double[] X) {
        //     int rank;
        //     double x = X[0]; double y = X[1];
        //     if (x < 0.5 && y < 0.5)
        //         rank = 0;
        //     else if (x >= 0.5 && y < 0.5)
        //         rank = 1;
        //     else if (x < 0.5 && y >= 0.5)
        //         rank = 2;
        //     else 
        //         rank = 3;

        //     return rank;
        // });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Creating database 'C:\BoSSS-localJobs\RisingBubble3D'.
Grid Edge Tags changed.
An equivalent grid (1a4702c1-21b5-433b-99b1-87acbc014d40) is already present in the database -- the grid will not be saved.


## Initial Values

In [15]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.25;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.5).Pow2() + (X[1] - 0.5).Pow2() + (X[2] - 0.5).Pow2()) - dropRadius; " + 
    "}");

In [16]:
var GravityValue = new Formula(
    "grav",
    false,
    "using ilPSP.Utils; " + 
    "double grav(double[] X) { return -0.98; }");

## Physical Settings

In [17]:
string testcase = "testcase1";

In [18]:
double rhoA = 1.0;      // bubble
double rhoB = 1.0;   // surrounding fluid
double muA = 1.0;
double muB = 1.0;
double sigma = 0.0;

switch(testcase) {
    case "testcase1": {
        rhoA = 100.0;
        rhoB = 1000.0;
        muA = 1.0;
        muB = 10.0;
        sigma = 24.5;

        break;
    }
    case "testcase2": {
        rhoA = 1.0;
        rhoB = 1000.0;
        muA = 0.1;
        muB = 10.0;
        sigma = 1.96;
        
        break;
    }
}


In [19]:
int k = 3;
int AMRlevel = 0;
double hmin = (1.0 / (double)Resolutions[0]) / ((double)AMRlevel + 1);
double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(rhoA, rhoB, sigma, hmin, k+1);
dt_sigma

0.01056655405839857

In [20]:
double dt = 0.01;

In [21]:
int numTs = (int)(3.0/dt);
numTs

300

## Control settings

In [22]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [23]:
restartSession.Timesteps.Last()

 { Time-step: 0; Physical time: 0s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }

In [24]:
int restartTS = restartSession.Timesteps.Last().TimeStepNumber.MajorNumber - 0;
restartTS

0

In [25]:
for (int idx = 0; idx < Grids.Length; idx++) {

    XNSE_Control C = new XNSE_Control();

    C.SetDGdegree(k);
    C.FieldOptions.Add(VariableNames.GravityZ + "#A", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });
    C.FieldOptions.Add(VariableNames.GravityZ + "#B", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });

    // physical parameters
    C.PhysicalParameters.rho_A = rhoA;
    C.PhysicalParameters.mu_A = muA;
    
    C.PhysicalParameters.rho_B = rhoB;
    C.PhysicalParameters.mu_B = muB;
    
    C.PhysicalParameters.Sigma = sigma;
    
    C.PhysicalParameters.IncludeConvection = true;

    
    if (restartRun) {
        C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, restartTS);
    } else {
        // set grid
        C.SetGrid(Grids[idx]);
    
        // initial conditions   
        C.AddInitialValue("Phi", PhiFunc);   
         
        C.AddInitialValue("GravityZ#A", GravityValue);
        C.AddInitialValue("GravityZ#B", GravityValue);
    }
    
    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;
 
    int ReInitP = 30;
    C.ReInitPeriod = ReInitP;

    // C.SkipSolveAndEvaluateResidual = true;

    //C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    //C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.MaxSolverIterations = 50;

    // C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
    C.dtFixed = dt;
    C.NoOfTimesteps = numTs;

    if (AMRlevel > 0) {
        C.AdaptiveMeshRefinement = true;
        C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel });
        C.AMR_startUpSweeps = AMRlevel;
    }

    // C.PostprocessingModules.Add(new EnergyLogging());
    C.TracingNamespaces = "BoSSS.Solution.AdvancedSolvers";
   
    if (restartRun) {
        C.SessionName = restartName;
    } else {
        int res = Resolutions[idx];
        C.SessionName = $"RB3D_fullDomain_{res}x{res}x{2*res}AMR{AMRlevel}_k{k}_{testcase}_ReInit{ReInitP}";
        //C.SessionName = $"RB3D_quarterDomain_{res}x{res}x{2*res}AMR{AMRlevel}_k{k}_{testcase}_setttingsCheck";
    }
    
    // C.GridPartType = GridPartType.Predefined;
    // C.GridPartOptions = "FourProcSplit_fullDomain";
    
    Controls.Add(C);
}

In [26]:
Controls.Select(C => C.SessionName)

[ RB3D_fullDomain_8x8x16AMR0_k3_testcase1_ReInit30 ]

## Launch Jobs

In [27]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 4;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.Activate(myBatch);
}

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job RB3D_fullDomain_8x8x16AMR0_k3_testcase1_ReInit30 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\RisingBubble3D-XNSE_Solver2025Jul09_145108.427429
copied 51 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
